# 06 - Agents

Demonstrate the AIMU agent support: single agents that autonomously loop over tool calls, multi-step workflows that chain agents, and streaming output.

## Setup

Create an `OllamaClient` with a tool-capable model and attach an `MCPClient` with a few demo tools. The same setup is used across all sections.

In [ ]:
import datetime
import nest_asyncio
from fastmcp import FastMCP

from aimu.models.ollama import OllamaClient, OllamaModel
from aimu.tools import MCPClient

# Required to allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

# --- MCP tools ---

mcp = FastMCP("Demo Tools")


@mcp.tool()
def get_current_date_and_time() -> str:
    """Returns the current date and time."""
    return str(datetime.datetime.now())


@mcp.tool()
def get_weather(location: str) -> str:
    """Returns the current weather for a given location."""
    # Stubbed for demo purposes
    return f"Sunny, 22°C in {location}"


@mcp.tool()
def calculate(expression: str) -> str:
    """Evaluates a simple arithmetic expression and returns the result."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"


# --- Model client ---

model_client = OllamaClient(OllamaModel.QWEN_3_8B)
model_client.mcp_client = MCPClient(server=mcp)

print("Model:", model_client.model.value)
print("Tools:", [t.name for t in model_client.mcp_client.list_tools()])

## A - Basic Agent

An `Agent` wraps a `ModelClient` and runs an agentic loop: after each `chat()` call, if the model used tools, the agent sends a continuation prompt to allow further tool use. The loop stops once the model responds without calling any tools.

This is the simplest possible usage — two lines.

In [ ]:
from aimu.agents import Agent

model_client.messages = []

agent = Agent(model_client, name="assistant", max_iterations=5)
result = agent.run("What is the current date and time, and what is 123 * 456?")

print(result)

The agent called both tools — date/time and calculator — and assembled the results into a final answer. Inspect the full message history to see every step.

In [ ]:
model_client.messages

## B - Agent with System Message

Pass a `system_message` to give the agent a specific persona or set of instructions.

In [ ]:
model_client.messages = []

agent = Agent(
    model_client,
    name="weather-bot",
    max_iterations=5,
)
model_client.system_message = (
    "You are a travel assistant. Always check the weather before giving advice. "
    "Be concise."
)

result = agent.run("Should I visit Paris or London today?")
print(result)

## C - Agent from Config

`Agent.from_config()` creates an agent from a plain dict. This makes it easy to define agents in configuration files or environment settings without writing boilerplate code.

In [ ]:
agent_config = {
    "name": "calculator-agent",
    "system_message": "You are a maths assistant. Use the calculate tool for all arithmetic.",
    "max_iterations": 8,
    "continuation_prompt": "Continue solving the problem step by step.",
}

model_client.messages = []
agent = Agent.from_config(agent_config, model_client)

print(f"Agent name:       {agent.name}")
print(f"Max iterations:   {agent.max_iterations}")
print(f"System message:   {model_client.system_message}")

In [ ]:
result = agent.run("What is (17 * 23) + (88 / 4)?")
print(result)

## D - Streaming Agent Output

`run_streamed()` yields `AgentChunk` objects — each with `agent_name`, `iteration`, `phase`, and `content`. This mirrors the `StreamChunk` API from `ModelClient.chat_streamed()`, extended with agent context.

The `iteration` field shows which loop round produced each chunk, making it easy to follow multi-step reasoning.

In [ ]:
from aimu.agents import AgentChunk
from aimu.models import StreamPhase

model_client.messages = []
model_client.system_message = None

agent = Agent(model_client, name="streamer", max_iterations=6)

current_iteration = -1

for chunk in agent.run_streamed("What is the weather in Tokyo, and what is 99 * 11?"):
    if chunk.iteration != current_iteration:
        current_iteration = chunk.iteration
        print(f"\n--- iteration {current_iteration} ---")

    if chunk.phase == StreamPhase.THINKING:
        print(f"[thinking] {chunk.content}", end="", flush=True)
    elif chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"[tool: {chunk.content['name']}] → {chunk.content['response']}")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## E - Multi-Step Workflow

A `Workflow` chains agents sequentially: the text output of each step becomes the input to the next. Each agent runs its own agentic loop independently.

Here a three-step pipeline: a **researcher** gathers facts using tools, a **analyst** interprets them, and a **formatter** produces a clean summary.

In [ ]:
from aimu.agents import Workflow


def make_client(cfg):
    """Factory used by Workflow.from_config to create a ModelClient per step."""
    client = OllamaClient(OllamaModel.QWEN_3_8B, system_message=cfg.get("system_message"))
    client.mcp_client = MCPClient(server=mcp)
    return client


workflow_configs = [
    {
        "name": "researcher",
        "system_message": (
            "You are a research assistant. Use available tools to collect facts. "
            "Output only the raw facts you found, one per line."
        ),
        "max_iterations": 6,
    },
    {
        "name": "analyst",
        "system_message": (
            "You are a data analyst. Interpret the provided facts and draw conclusions. "
            "Be concise."
        ),
        "max_iterations": 3,
    },
    {
        "name": "formatter",
        "system_message": (
            "You are a copywriter. Rewrite the provided analysis as a short, friendly "
            "paragraph suitable for a general audience. No bullet points."
        ),
        "max_iterations": 1,
    },
]

wf = Workflow.from_config(workflow_configs, make_client)

result = wf.run("Gather the current date/time and the weather in London, then summarise.")
print(result)

## F - Streaming Workflow Output

`Workflow.run_streamed()` yields `WorkflowChunk` objects that extend `AgentChunk` with a `step` index. This lets you track progress across the full pipeline in real time.

In [ ]:
from aimu.agents import WorkflowChunk

wf2 = Workflow.from_config(workflow_configs, make_client)

current_step = -1

for chunk in wf2.run_streamed("Get the current time and weather in Sydney, then summarise."):
    if chunk.step != current_step:
        current_step = chunk.step
        agent_name = workflow_configs[current_step]["name"]
        print(f"\n=== Step {current_step}: {agent_name} ===")

    if chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"  [tool: {chunk.content['name']}] → {chunk.content['response']}")
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## G - Clean Up

Delete model clients to release any held resources.

In [ ]:
del model_client, wf, wf2